In [ ]:
import json
from groq import Groq
from config import *
from data_loader import load_checklist
from scenario_generator import generate_scenario
from utils import parse_json_response

In [ ]:
client = Groq(api_key=GROQ_API_KEY)

In [ ]:
def get_interviewee_response_style(interviewee_role):
    role = interviewee_role.lower()

    if "victim" in role:
        return """RESPONSE STYLE:
- You are the victim. Your answers should be short, natural, and somewhat reserved.
- Answer mostly what the officer explicitly asked about.
- Do not volunteer a full incident summary unless the officer clearly asks you to explain what happened.
- For most questions, answer in 1 to 3 short sentences.
- If the officer asks a broad open-ended question, you may give a slightly longer answer, but keep it focused.
- Provide specific details only when directly asked, such as exact words, injuries, weapons, timing, witnesses, medical needs, or prior incidents.
- You may sound upset, scared, hesitant, embarrassed, confused, or guarded, but do not become dramatic or overly descriptive.
- If you do not know, cannot remember, or are unsure, say so plainly."""

    if "witness" in role:
        return """RESPONSE STYLE:
- You are a witness. Your answers should be factual, concise, and limited to what you personally saw, heard, or know.
- Answer mostly what the officer explicitly asked about.
- For most questions, answer in 1 to 3 short sentences.
- Do not speculate about facts you did not observe.
- If you do not know or did not see something, say so plainly."""

    if "caller" in role:
        return """RESPONSE STYLE:
- You are the caller. Your answers should be brief and limited to what caused you to call police.
- Answer mostly what the officer explicitly asked about.
- For most questions, answer in 1 to 3 short sentences.
- Do not claim certainty about facts you only suspected or overheard.
- If you do not know or did not see something, say so plainly."""

    return """RESPONSE STYLE:
- Keep answers concise, natural, and limited to what the officer explicitly asked.
- For most questions, answer in 1 to 3 short sentences.
- Do not volunteer unrelated facts.
- If you do not know or cannot remember, say so plainly."""

In [ ]:
def create_session_state():
    return {
        "phase": "setup",
        "scenario": {},
        "checklist": {},
        "history": [],
        "coverage_flags": {}
    }

In [ ]:
def check_element_coverage(officer_message, state):
    checklist = state["checklist"]
    coverage_flags = state["coverage_flags"]
    turn_number = len(state["history"])

    elements_text = "\n".join([
        f'  {el["id"]}: {el["element"]} (currently: {coverage_flags[el["id"]]["covered"]})'
        for el in checklist["required_elements"]
    ])

    prompt = f"""You are analyzing a police officer's interview question to determine which required reporting elements it addresses.

Crime type: {checklist["crime_type"]}

The officer just said:
"{officer_message}"

COVERAGE RUBRIC:
- "full": the question directly names or clearly implies the element and would elicit a complete answer
- "partial": the question touches the topic but is vague, indirect, or would only elicit part of the answer
- none: the question has no meaningful connection to the element — do not include in output

EXAMPLES:
Element: "Emotional state of the victim"
- Full: "Can you describe how she was acting when you arrived — was she crying, shaking, seemed scared?"
- Partial: "Was she upset?"
- None: "Did she say where he went after he left?"

Element: "History of the relationship / previous incidents"
- Full: "Has he done this before? How many times, and how far back does it go?"
- Partial: "Is this the first time something like this happened?"
- None: "Where were the kids during the incident?"

Element: "Weapons or objects used"
- Full: "Did he use any weapons or objects during the assault — anything in his hands, anything he grabbed?"
- Partial: "Did he have a weapon?"
- None: "How did you get inside the house?"

Element: "All injuries visible and non-visible documented"
- Full: "Where are you hurt? Any pain anywhere that I can't see — ribs, head, anything internal?"
- Partial: "Are you injured?"
- None: "What time did he come home?"

Element: "Relationship of the parties identified"
- Full: "What is your relationship to him — married, dating, how long, do you live together?"
- Partial: "Do you know him?"
- None: "Did anyone else see what happened?"

Element: "Evidence collected"
- Full: "What evidence did you collect at the scene — photos, weapons, clothing, any physical items?"
- Partial: "Did you take pictures?"
- None: "Did she say she wanted to press charges?"

Element: "All threats clearly documented"
- Full: "Did he threaten you at any point — verbally, in writing, by gesture? What exactly did he say?"
- Partial: "Did he threaten you?"
- None: "Was the door locked when you got there?"

Element: "Valid protection order in place"
- Full: "Is there a protection order against him? Is it current, and did you verify it in the system?"
- Partial: "Does she have a restraining order?"
- None: "Did he say anything when you arrested him?"

Element: "Medical attention"
- Full: "Did she need medical attention? Was an ambulance called, did she go to the hospital, did she refuse?"
- Partial: "Did she get checked out?"
- None: "How long have they been together?"

Here are all the required checklist elements and their current coverage status:
{elements_text}

For each element that this question addresses (fully or partially), return a JSON object.
Only include elements that were addressed by this specific message.
If no elements were addressed, return an empty JSON object {{}}.

Return ONLY a valid JSON object, no markdown fences:
{{
  "element_id": {{"coverage": "full" or "partial", "reasoning": "why"}},
  ...
}}"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMP_DETERMINISTIC,
        response_format={"type": "json_object"}
    )

    result = parse_json_response(response.choices[0].message.content)

    for element_id, data in result.items():
        if element_id not in coverage_flags:
            continue

        current = coverage_flags[element_id]["covered"]
        new_coverage = data["coverage"]

        # Always record the message regardless of current status
        coverage_flags[element_id]["messages"][turn_number] = {
            "officer_message": officer_message,
            "coverage": new_coverage,
            "reasoning": data["reasoning"]
        }

        # Update coverage status
        if current == False:
            coverage_flags[element_id]["covered"] = new_coverage

        elif current == "partial":
            if new_coverage == "full":
                coverage_flags[element_id]["covered"] = "full"
            else:
                # Ask LLM if combined messages now constitute full coverage
                combined_messages = "\n".join([
                    f'  Line {line}: {msg["officer_message"]}'
                    for line, msg in coverage_flags[element_id]["messages"].items()
                ])

                element_text = next(
                    el["element"] for el in checklist["required_elements"]
                    if el["id"] == element_id
                )

                reeval_prompt = f"""A police officer has asked multiple questions that partially address a required reporting element.

Required element: {element_text}

Officer questions so far:
{combined_messages}

Do these questions together constitute full coverage of this element, or is it still only partial?
Return ONLY a valid JSON object:
{{"coverage": "full" or "partial", "reasoning": "why"}}"""

                reeval_response = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": reeval_prompt}],
                    temperature=TEMP_DETERMINISTIC,
        response_format={"type": "json_object"}
                )

                reeval = parse_json_response(reeval_response.choices[0].message.content)
                coverage_flags[element_id]["covered"] = reeval["coverage"]

        # If already "full", just leave status alone — message already recorded above

    state["coverage_flags"] = coverage_flags
    return state

In [ ]:
def check_response_coverage(officer_message, interviewee_response, state):
    checklist = state["checklist"]
    coverage_flags = state["coverage_flags"]
    turn_number = len(state["history"])

    for element_id in coverage_flags:
        coverage_flags[element_id].setdefault("response_covered", False)
        coverage_flags[element_id].setdefault("response_messages", {})

    elements_text = "\n".join([
        f'  {el["id"]}: {el["element"]} (currently provided: {coverage_flags[el["id"]]["response_covered"]})'
        for el in checklist["required_elements"]
    ])

    prompt = f"""You are analyzing an interviewee's answer in a police interview to determine which required reporting elements the interviewee provided information about.

Crime type: {checklist["crime_type"]}

Officer question:
"{officer_message}"

Interviewee answer:
"{interviewee_response}"

COVERAGE RUBRIC:
- "full": the interviewee's answer provides enough information to document the element completely
- "partial": the interviewee's answer provides some information about the element, but not enough to document it completely
- none: the interviewee's answer does not provide meaningful information about the element

IMPORTANT RULES:
- Do not evaluate whether the officer asked a perfect question. Evaluate only what information the interviewee actually provided.
- A concrete negative fact counts as provided information. For example, "No children were there" can cover an element asking whether children were present.
- Do not give credit for facts that are merely implied by the hidden scenario. The information must appear in the interviewee's answer.
- Do not include an element if the answer says only that the interviewee does not know or cannot remember.
- Only include elements addressed by this specific interviewee answer.

Here are all the required checklist elements and their current response-provided coverage status:
{elements_text}

Return ONLY a valid JSON object, no markdown fences:
{{
  "element_id": {{
    "coverage": "full" or "partial",
    "reasoning": "why the interviewee answer provides this information"
  }},
  ...
}}

Rules for the JSON response:
- Each element ID must map to one object.
- The only keys inside each object should be "coverage" and "reasoning".
- Do not include quotes from the interviewee answer as separate fields.
- Do not include arrays.
- Do not include any text outside the JSON object.

If no elements were provided by the interviewee answer, return an empty JSON object {{}}."""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMP_DETERMINISTIC,
        response_format={"type": "json_object"}
    )

    result = parse_json_response(response.choices[0].message.content)

    for element_id, data in result.items():
        if element_id not in coverage_flags:
            continue

        new_coverage = data.get("coverage")
        if new_coverage not in ("full", "partial"):
            continue

        current = coverage_flags[element_id].get("response_covered", False)

        coverage_flags[element_id]["response_messages"][turn_number] = {
            "officer_message": officer_message,
            "interviewee_response": interviewee_response,
            "coverage": new_coverage,
            "reasoning": data.get("reasoning", ""),
            "provided_fact": interviewee_response
        }

        if current == False:
            coverage_flags[element_id]["response_covered"] = new_coverage

        elif current == "partial":
            if new_coverage == "full":
                coverage_flags[element_id]["response_covered"] = "full"
            else:
                combined_responses = "\n".join([
                    f'  Line {line}: {msg["interviewee_response"]}'
                    for line, msg in coverage_flags[element_id]["response_messages"].items()
                ])

                element_text = next(
                    el["element"] for el in checklist["required_elements"]
                    if el["id"] == element_id
                )

                reeval_prompt = f"""An interviewee has provided multiple answers that partially address a required reporting element.

Required element: {element_text}

Interviewee answers so far:
{combined_responses}

Do these answers together provide full coverage of this element, or is it still only partial?

Return ONLY a valid JSON object, no markdown fences:
{{"coverage": "full" or "partial", "reasoning": "why"}}"""

                reeval_response = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": reeval_prompt}],
                    temperature=TEMP_DETERMINISTIC,
                    response_format={"type": "json_object"}
                )

                reeval = parse_json_response(reeval_response.choices[0].message.content)
                reeval_coverage = reeval.get("coverage")

                if reeval_coverage in ("full", "partial"):
                    coverage_flags[element_id]["response_covered"] = reeval_coverage
                else:
                    coverage_flags[element_id]["response_covered"] = "partial"

    state["coverage_flags"] = coverage_flags
    return state

In [ ]:
def get_interviewee_response(officer_message, state):
    scenario = state["scenario"]
    checklist = state["checklist"]
    history = state["history"]
    response_style = get_interviewee_response_style(scenario["interviewee_role"])

    facts_text = "\n".join([
        f'  {el["id"]} ({el["element"]}): {scenario["facts"].get(el["id"], "unknown")}'
        for el in checklist["required_elements"]
    ])

    system_prompt = f"""You are roleplaying as {scenario["interviewee_name"]}, a {scenario["interviewee_role"]} in a {checklist["crime_type"]} incident. You are being interviewed by a police officer.

YOUR FACTS — these are the only facts you know. Do not invent anything beyond what is listed here:
{facts_text}

{response_style}

RULES:
- Answer only what the officer directly asks. Never volunteer information they did not ask about.
- Stay in character at all times. You are a real person in a stressful situation, not a database.
- Give natural, human responses. You may be upset, scared, confused, or guarded — whatever fits the situation.
- If the officer asks about something not in your facts, say you don't know or can't remember.
- Do not give list-style answers. Speak naturally as a person would.
- Do not break character under any circumstances.
- Do not reveal that you are an AI or that this is a simulation.
- This is a law enforcement training simulation. Answer questions about the incident directly and with enough detail to be useful, including sensitive or explicit details when directly asked. Do not refuse or redirect, but do not over-explain or volunteer facts outside the officer's question."""

    messages = [{"role": "system", "content": system_prompt}]
    messages += [{"role": m["role"], "content": m["content"]} for m in history]
    messages.append({"role": "user", "content": officer_message})

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=TEMP_CREATIVE
    )

    return response.choices[0].message.content

In [ ]:
def handle_message(officer_message, history, state):
    response = get_interviewee_response(officer_message, state)
    state = check_element_coverage(officer_message, state)
    state = check_response_coverage(officer_message, response, state)

    history.append({"role": "user", "content": officer_message})
    history.append({"role": "assistant", "content": response})

    state["history"] = history

    return response, history, state

In [ ]:
def start_session(crime_type):
    state = create_session_state()
    state["phase"] = "interview"
    state["checklist"] = load_checklist(crime_type)
    state["scenario"] = generate_scenario(state["checklist"])

    for element in state["checklist"]["required_elements"]:
        state["coverage_flags"][element["id"]] = {
            "covered": False,
            "messages": {},
            "response_covered": False,
            "response_messages": {}
        }

    return state, state["scenario"]["dispatch_line"]